### Basic RAG application
**Goal**

- Build a RAG application which reads PDF and query LLM

**Steps**
- Read PDF file from url
- Split, embed and store it in vector store
- Create chain manually and with LCEL
- Ask a question
  
**Tech stack**
- LangChain
- FAISS
- Ollama

### Import libraries

In [5]:
from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.output_parsers import StrOutputParser

from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain

### Get model and embeddings

In [6]:
llm = ChatOllama(
    model="tinyllama", 
    temperature=0.3, 
    num_predict=300, 
    num_ctx=2048
)
embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

### Download pdf from url

In [7]:
PDF_URL = "https://arxiv.org/pdf/1810.04805"
loader = PyPDFLoader(PDF_URL)
pages = loader.load()

for page in pages:
    page.metadata["source"] = PDF_URL

print(f"Loaded {len(pages)} pages from: {PDF_URL}")
print(f"metadata: {pages[0].metadata}")
print(f"content : {pages[0].page_content[:50].strip()}")

Loaded 16 pages from: https://arxiv.org/pdf/1810.04805
metadata: {'producer': 'pdfTeX-1.40.17', 'creator': 'LaTeX with hyperref package', 'creationdate': '2019-05-28T00:07:51+00:00', 'author': '', 'keywords': '', 'moddate': '2019-05-28T00:07:51+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.17 (TeX Live 2016) kpathsea version 6.2.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'https://arxiv.org/pdf/1810.04805', 'total_pages': 16, 'page': 0, 'page_label': '1'}
content : BERT: Pre-training of Deep Bidirectional Transform


### Split documents

In [8]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, 
    chunk_overlap=50, 
    separators=["\n\n", "\n", ". ", " "],
)
chunks = splitter.split_documents(pages)

print(f"{len(pages)} pages → {len(chunks)} chunks")
print(f"Avg chunk size: {sum(len(c.page_content) for c in chunks) // len(chunks)} chars")
print(f"Sample chunk  : {chunks[0].page_content[:120].strip()}")

16 pages → 152 chunks
Avg chunk size: 449 chars
Sample chunk  : BERT: Pre-training of Deep Bidirectional Transformers for
Language Understanding
Jacob Devlin Ming-Wei Chang Kenton Lee


### Store in vectorstore and create retriever

In [9]:
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

In [10]:
test = retriever.invoke("contrast ratio requirements")
for doc in test:
    print(f"[page {doc.metadata.get('page', '?')}] {doc.page_content[:80].strip()}")

[page 12] of possible values to work well across all tasks:
• Batch size: 16, 32
13https:/
[page 6] that contain at least one of the provided possible answers.
System Dev Test
ESIM
[page 15] proaches, as we expect the mismatch will be am-
pliﬁed for the feature-based app
[page 6] 5 Ablation Studies
In this section, we perform ablation experiments
over a numbe


### Prompt

In [11]:
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a helpful assistant that answers questions about documents. "
        "Answer ONLY from the provided context. "
        "If the context does not contain the answer, say: "
        "'The document does not contain enough information to answer this.' "
        "Cite the page number when possible. Be concise.\n\n"
        "Context:\n{context}"
    ),
    (
        "human",
        "{input}"
    ),
])

### Create chains

In [12]:
document_chain = create_stuff_documents_chain(
    prompt=prompt,
    llm=llm,
)

In [13]:
rag_chain = create_retrieval_chain(
    combine_docs_chain=document_chain,
    retriever=retriever,
)

### Ask a question

In [14]:
def ask(question: str):
    print(f"Q: {question}")
    response = rag_chain.invoke({"input": question})
    print(response.keys())
    print(f"A: {response['answer'].strip()}")

    # Show which pages were used
    pages_used = list(dict.fromkeys(
        str(doc.metadata.get("page", "?"))
        for doc in response["context"]
    ))
    print(f"Retrieved from pages: {pages_used}")

In [15]:
ask("What is the purpose of this document?")

Q: What is the purpose of this document?
dict_keys(['input', 'context', 'answer'])
A: The purpose of this document is to provide a detailed explanation of the context, the problem, and the solution for a given research question. It includes the background information, the literature review, the research methodology, the results, and the discussion. The document is intended to be used as a reference for the researcher and to provide insights into the research question and its potential solutions.
Retrieved from pages: ['6', '11', '15']


## With LCEL

In [16]:
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

In [17]:
def format_docs(docs):
    return "\n\n---\n\n".join(
        f"[Page {doc.metadata.get('page', '?')}]\n{doc.page_content}"
        for doc in docs
    )

### Create chains

In [18]:
rag_chain = (
    RunnableParallel({
        "context": retriever | format_docs,
        "input": RunnablePassthrough(),
    })
    | prompt
    | llm
    | StrOutputParser()
)

In [19]:
retrieve_chain = RunnableParallel({
    "context": retriever | format_docs,
    "input": RunnablePassthrough(),
})
retrieved = retrieve_chain.invoke("What is BERT pre-training?")

In [20]:
answer_chain = prompt | llm | StrOutputParser()

### Ask a question

In [21]:
def ask_pipe_version(question: str):
    print(f"Q: {question}")

    # Simple version - just answer string
    answer = rag_chain.invoke(question)
    print(f"A: {answer.strip()}")

    # Also get source pages using retrieve_chain separately
    retrieved = retrieve_chain.invoke(question)
    # retrieved["context"] is the formatted string - parse pages from it
    source_docs = retriever.invoke(question)
    pages_used = list(dict.fromkeys(
        str(doc.metadata.get("page", "?")) for doc in source_docs
    ))
    print(f"Retrieved from pages: {pages_used}\n")

In [22]:
ask_pipe_version("What is BERT pre-training?")

Q: What is BERT pre-training?
A: BERT (BiDirectional Encoder Representations from Transformers) is a new language representation model designed by the authors of the paper "Pre-training of Deep Bidirectional Transformer for Language Understanding" (Peter et al., 2018a). The BERT model is designed to pre-train a transformer encoder-decoder model that can represent a sentence or a paragraph in a bi-directional manner. The model is designed to pre-train the encoder and decoder separately, and to fine-tune the model on downstream tasks such as question answering (QA), natural language inference (NLI), and sentiment analysis (SQuAD). BERT is a transformer-based model that uses a Bi-Directional Encoder Representation (BERT) to encode the input sentence or paragraph, and a Bi-Directional Decoder (BID) to decode the encoded representation back into the original sentence or paragraph. The BEBER model is trained on a large dataset of unlabelled sentences, and is evaluated on a variety of downstr